#### Built-in SQL Functions
- String functions
- Date / Timestamp functions
- Numeric functions
- Type casting
- NULL handling (IFNULL, NULLIF, COALESCE, NVL)
- Conditional expressions (CASE WHEN, IF, IIF)
- Collection functions (array, map)

In [0]:
"""
QUICK REFERENCE:
─────────────────────────────────────────────────────────────
NULL FUNCTIONS:
  COALESCE(a, b, c)   → first non-null among a, b, c
  IFNULL(a, b)        → b if a is null  (= NVL)
  NVL2(a, b, c)       → b if a is not null, else c
  NULLIF(a, b)        → NULL if a == b, else a
  TRY_CAST            → NULL on cast failure (vs exception)

DATE GOTCHAS:
  DATEDIFF(end, start)         → days between (end - start)
  MONTHS_BETWEEN(end, start)   → months (can be fractional)
  DATE_FORMAT uses Java patterns: yyyy MM dd HH mm ss
  TO_DATE  → string → date
  TO_TIMESTAMP → string → timestamp

EXPLODE vs POSEXPLODE:
  EXPLODE(arr)         → one row per element
  POSEXPLODE(arr)      → (pos, element) per row
  EXPLODE_OUTER        → keeps row even if array is null/empty
─────────────────────────────────────────────────────────────
"""


##### 1. STRING FUNCTIONS

In [0]:
from pyspark.sql import SparkSession

# ---------------------------------------------------------------------------
# Sample data
# ---------------------------------------------------------------------------
data = [
    (1, "  Alice Johnson  ", "Engineering", 95000.567, "2020-03-15", "2024-01-10 08:30:00"),
    (2, "bob smith",        "Marketing",   72000.0,   "2019-07-01", "2024-01-11 14:45:30"),
    (3, "CAROL WHITE",      "Engineering", 110000.1,  "2018-11-20", "2024-01-12 09:00:00"),
    (4, "Dave",             "HR",          60000.75,  "2021-01-10", "2024-01-13 17:20:15"),
    (5, "eve",              None,          85000.0,   "2020-09-25", "2024-01-14 12:00:00"),
]
schema = "id INT, name STRING, department STRING, salary DOUBLE, hire_date STRING, last_login STRING"
df = spark.createDataFrame(data, schema=schema)
df.createOrReplaceTempView("employees")

# ---------------------------------------------------------------------------
# 1. STRING FUNCTIONS
# ---------------------------------------------------------------------------
print("=== 1. String Functions ===")
spark.sql("""
    SELECT
        name,
        TRIM(name)                        AS trimmed,
        LTRIM(name)                       AS left_trimmed,
        UPPER(TRIM(name))                 AS upper_name,
        LOWER(TRIM(name))                 AS lower_name,
        INITCAP(LOWER(TRIM(name)))        AS proper_case,
        LENGTH(TRIM(name))                AS name_length,
        SUBSTR(TRIM(name), 1, 5)          AS first_5,
        SUBSTRING(TRIM(name), 1, 5)       AS first_5_alt,
        INSTR(TRIM(name), ' ')            AS space_position,
        REPLACE(TRIM(name), ' ', '_')     AS underscored,
        LPAD(CAST(id AS STRING), 4, '0')  AS padded_id,
        RPAD(department, 15, '.')         AS padded_dept,
        CONCAT(id, ' - ', TRIM(name))     AS label,
        CONCAT_WS(' | ', id, TRIM(name), department) AS piped
    FROM employees
    LIMIT 3
""").show(truncate=False)

# SPLIT and REGEXP
print("=== 1b. SPLIT, LIKE, REGEXP ===")
spark.sql("""
    SELECT
        TRIM(name)                                    AS name,
        SPLIT(TRIM(name), ' ')                        AS name_parts,
        SPLIT(TRIM(name), ' ')[0]                     AS first_name,
        try_element_at(SPLIT(TRIM(name), ' '), 2)         AS last_name, -- returns NULL if no second element
        name LIKE '%on%'                              AS contains_on,
        name RLIKE '^\\s*[Aa]'                        AS starts_with_a,
        REGEXP_EXTRACT(TRIM(name), '^(\\w+)', 1)      AS first_word,
        REGEXP_REPLACE(TRIM(name), '\\s+', '-')       AS hyphenated
    FROM employees
""").show(truncate=False)


##### 2. DATE & TIMESTAMP FUNCTIONS 

In [0]:
# ---------------------------------------------------------------------------
# 2. DATE & TIMESTAMP FUNCTIONS 
# ---------------------------------------------------------------------------
print("=== 2. Date Functions ===")
spark.sql("""
    SELECT
        hire_date,
        TO_DATE(hire_date, 'yyyy-MM-dd')              AS parsed_date,
        YEAR(hire_date)                               AS yr,
        MONTH(hire_date)                              AS mo,
        DAY(hire_date)                                AS dy,
        DAYOFWEEK(hire_date)                          AS dow,       -- 1=Sun
        DAYOFYEAR(hire_date)                          AS doy,
        WEEKOFYEAR(hire_date)                         AS woy,
        QUARTER(hire_date)                            AS qtr,
        DATE_FORMAT(hire_date, 'MMMM dd, yyyy')       AS formatted,
        DATE_ADD(hire_date, 90)                       AS plus_90d,
        DATE_SUB(hire_date, 30)                       AS minus_30d,
        ADD_MONTHS(hire_date, 6)                      AS plus_6mo,
        DATEDIFF(CURRENT_DATE(), hire_date)           AS days_since_hire,
        MONTHS_BETWEEN(CURRENT_DATE(), hire_date)     AS months_since,
        LAST_DAY(hire_date)                           AS last_of_month,
        NEXT_DAY(hire_date, 'Monday')                 AS next_monday,
        TRUNC(hire_date, 'MM')                        AS month_start,
        TRUNC(hire_date, 'YEAR')                      AS year_start
    FROM employees
    LIMIT 3
""").show(truncate=False)

print("=== 2b. Timestamp Functions ===")
spark.sql("""
    SELECT
        last_login,
        TO_TIMESTAMP(last_login, 'yyyy-MM-dd HH:mm:ss')   AS ts,
        UNIX_TIMESTAMP(last_login, 'yyyy-MM-dd HH:mm:ss') AS epoch,
        FROM_UNIXTIME(
            UNIX_TIMESTAMP(last_login,'yyyy-MM-dd HH:mm:ss')
        )                                                  AS back_to_ts,
        HOUR(last_login)                                   AS hr,
        MINUTE(last_login)                                 AS min,
        SECOND(last_login)                                 AS sec,
        DATE_TRUNC('hour', last_login)                     AS trunc_hour,
        DATE_TRUNC('day',  last_login)                     AS trunc_day,
        CURRENT_TIMESTAMP()                                AS now,
        CURRENT_DATE()                                     AS today
    FROM employees
    LIMIT 3
""").show(truncate=False)


##### 3. NUMERIC FUNCTIONS

In [0]:
# ---------------------------------------------------------------------------
# 3. NUMERIC FUNCTIONS
# ---------------------------------------------------------------------------
print("=== 3. Numeric Functions ===")
spark.sql("""
    SELECT
        salary,
        ROUND(salary, 1)          AS rounded_1,
        ROUND(salary, 0)          AS rounded_0,
        CEIL(salary)              AS ceiling,
        FLOOR(salary)             AS floor_val,
        ABS(-salary)              AS absolute,
        MOD(CAST(salary AS INT), 1000) AS modulo,
        SQRT(salary)              AS sqrt_val,
        POWER(2, 10)              AS two_to_10,
        LOG(10, salary)           AS log10,
        LOG2(salary)              AS log2_val,
        EXP(1)                    AS euler,
        PI()                      AS pi_val,
        GREATEST(salary, 80000)   AS at_least_80k,
        LEAST(salary, 90000)      AS at_most_90k,
        SIGN(salary - 80000)      AS sign_val  -- -1, 0, 1
    FROM employees
    LIMIT 3
""").show(truncate=False)

##### 4. TYPE CASTING

In [0]:
# ---------------------------------------------------------------------------
# 4. TYPE CASTING
#    CAST(expr AS type)  or  expr::type  (Databricks shorthand)
# ---------------------------------------------------------------------------
print("=== 4. Type Casting ===")
spark.sql("""
    SELECT
        id,
        CAST(id AS STRING)             AS id_str,
        CAST(salary AS INT)            AS salary_int,
        CAST(salary AS BIGINT)         AS salary_bigint,
        CAST('123.45' AS DOUBLE)       AS str_to_dbl,
        CAST(hire_date AS DATE)        AS date_cast,
        CAST(last_login AS TIMESTAMP)  AS ts_cast,
        CAST('true' AS BOOLEAN)        AS bool_val,
        TRY_CAST('abc' AS INT)         AS safe_cast  -- returns NULL on failure
    FROM employees
    LIMIT 2
""").show(truncate=False)


##### 5. NULL HANDLING

In [0]:
# ---------------------------------------------------------------------------
# 5. NULL HANDLING
#    COALESCE  → first non-null from list
#    IFNULL    → IFNULL(expr, replacement)    — 2-arg coalesce
#    NVL       → alias for IFNULL
#    NVL2      → NVL2(expr, if_not_null, if_null)
#    NULLIF    → returns NULL if two args are equal, else first arg
#    IS NULL / IS NOT NULL
# ---------------------------------------------------------------------------
print("=== 5. NULL Handling ===")
spark.sql("""
    SELECT
        id,
        name,
        department,
        COALESCE(department, 'Unknown')         AS dept_coalesce,
        IFNULL(department, 'Unknown')           AS dept_ifnull,
        NVL(department, 'Unknown')              AS dept_nvl,
        NVL2(department, 'Has dept', 'No dept') AS dept_nvl2,
        NULLIF(department, 'HR')                AS dept_nullif,  -- NULL if HR
        department IS NULL                      AS is_null,
        department IS NOT NULL                  AS is_not_null
    FROM employees
""").show(truncate=False)

##### 6. CONDITIONAL EXPRESSIONS

In [0]:
# ---------------------------------------------------------------------------
# 6. CONDITIONAL EXPRESSIONS
#    CASE WHEN ... THEN ... ELSE ... END
#    IF(condition, true_val, false_val)
# ---------------------------------------------------------------------------
print("=== 6. CASE WHEN & IF ===")
spark.sql("""
    SELECT
        name,
        salary,
        department,

        -- Searched CASE (most flexible)
        CASE
            WHEN salary >= 100000 THEN 'Senior'
            WHEN salary >= 80000  THEN 'Mid-level'
            WHEN salary >= 60000  THEN 'Junior'
            ELSE 'Entry'
        END AS seniority,

        -- Simple CASE (equality checks only)
        CASE department
            WHEN 'Engineering' THEN 'Tech'
            WHEN 'Marketing'   THEN 'Business'
            ELSE 'Other'
        END AS dept_group,

        -- IF (shorthand for 2-branch CASE)
        IF(salary > 80000, 'High earner', 'Standard') AS earner_type,

        -- Nested CASE
        CASE
            WHEN department IS NULL THEN 'No dept'
            WHEN department = 'Engineering' AND salary > 100000 THEN 'Top Eng'
            ELSE department
        END AS dept_label
    FROM employees
""").show(truncate=False)

##### 7. COLLECTION FUNCTIONS

In [0]:
# ---------------------------------------------------------------------------
# 7. COLLECTION FUNCTIONS (array, map, struct)
# ---------------------------------------------------------------------------
spark.sql("""
    CREATE OR REPLACE TEMP VIEW products AS
    SELECT * FROM VALUES
        (1, ARRAY('red','green','blue'),   MAP('price', 9.99,  'stock', 100.0)),
        (2, ARRAY('small','large'),        MAP('price', 49.99, 'stock', 50.0)),
        (3, ARRAY('cotton','polyester'),   MAP('price', 19.99, 'stock', 0.0))
    AS t(id, tags, attrs)
""")

print("=== 7. Array & Map Functions ===")
spark.sql("""
    SELECT
        id,
        tags,
        SIZE(tags)                AS tag_count,
        tags[0]                   AS first_tag,
        ARRAY_CONTAINS(tags,'red') AS has_red,
        SORT_ARRAY(tags)          AS sorted_tags,
        EXPLODE(tags)             AS tag_exploded,
        attrs,
        attrs['price']            AS price,
        MAP_KEYS(attrs)           AS attr_keys,
        MAP_VALUES(attrs)         AS attr_vals
    FROM products
""").show(truncate=False)


In [0]:
spark.stop()
